In [1]:
import easyocr
from PIL import ImageGrab
import numpy as np

# Create an EasyOCR Reader instance
reader = easyocr.Reader(['en'])  # 'en' stands for English language

def capture_screenshot():
    # Capture the full screen (you can modify the bbox to capture specific regions)
    screenshot = ImageGrab.grab()
    screenshot_np = np.array(screenshot)
    return screenshot_np

def ocr_read_text(image):
    # Use EasyOCR to detect and recognize text in the image
    result = reader.readtext(image)
    
    # Extract the text from the result
    text = ''
    for detection in result:
        text += detection[1] + '\n'
    
    return text

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


In [ ]:
from helper import alt_tab
import time

alt_tab()
ss = capture_screenshot()
txt = ocr_read_text(ss)
alt_tab()
print(txt)

Score: 0
S W A



In [27]:
import time
import pyautogui
import numpy as np
from PIL import ImageGrab
import easyocr
from PIL import Image
import re  # For regular expressions to filter and clean the text

# Alt+Tab Simulation to switch between windows
def alt_tab():
    pyautogui.keyDown('alt')  # Hold down the alt key
    pyautogui.press('tab')    # Press the tab key
    pyautogui.keyUp('alt')    # Release the alt key

# Capture a screenshot of the screen or a region
def capture_screenshot():
    # Capture the entire screen
    screenshot = ImageGrab.grab()  # Grab the entire screen
    return screenshot

# OCR Reading Text from Image using EasyOCR
def ocr_read_text(image):
    # Initialize EasyOCR reader
    reader = easyocr.Reader(['en'])  # You can add more languages if needed
    
    # Convert the screenshot to a numpy array (required by EasyOCR)
    image_np = np.array(image)
    
    # Use EasyOCR to read the text
    result = reader.readtext(image_np)
    
    # Extract the text from the result (list of detected texts)
    extracted_text = " ".join([text[1] for text in result])
    
    return extracted_text

# Resize image while maintaining aspect ratio
def resize_image(image, base_width):
    # Get current width and height of the image
    width, height = image.size
    aspect_ratio = height / width
    
    # Calculate the new height to maintain the aspect ratio
    new_height = int(base_width * aspect_ratio)
    
    # Resize image
    resized_image = image.resize((base_width, new_height))
    return resized_image

# Post-process OCR text to clean and standardize the output
def clean_ocr_output(txt):
    # Trim text to start from the first number
    match = re.search(r'\d', txt)  # Find the first digit
    if match:
        txt = txt[match.start():]  # Trim everything before the first digit
    
    # Remove spaces and replace '$' with 'S'
    txt = txt.replace(' ', '').replace('$', 'S')
    
    # Now create a list with each character in a single spot (keep digits as they are)
    cleaned_txt = []
    temp_num = ""  # Temporary string to accumulate multi-digit numbers
    
    for char in txt:
        if char.isdigit():
            temp_num += char  # Add digits to the temporary number string
        else:
            if temp_num:
                cleaned_txt.append(int(temp_num))  # Append the accumulated number to the list
                temp_num = ""  # Reset the temporary number string
            if char.isalpha():  # Only keep alphabetic characters
                cleaned_txt.append(char)  # Add 'W', 'A', 'S', 'D' as characters
    
    # If there's any leftover number at the end, add it to the list
    if temp_num:
        cleaned_txt.append(int(temp_num))
    
    # Convert 'W', 'A', 'S', 'D' to numbers
    char_to_number = {'W': 0, 'A': 1, 'S': 2, 'D': 3}
    for i in range(len(cleaned_txt)):
        if cleaned_txt[i] in char_to_number:
            cleaned_txt[i] = char_to_number[cleaned_txt[i]]
    
    return cleaned_txt



# Main flow: Switch window, capture screenshot, resize, and run OCR
alt_tab()  # Switch window
time.sleep(0.5)  # Reduced sleep time for faster switching

ss = capture_screenshot()  # Capture screenshot

# Resize to a smaller width while maintaining the aspect ratio
base_width = 240  # You can modify this base width for your needs
ss_resized = resize_image(ss, base_width)

# Run OCR on the resized image
start_time = time.time()
txt = ocr_read_text(ss_resized)
end_time = time.time()

# Clean and process the OCR text
cleaned_txt = clean_ocr_output(txt)

# Display results
print(f"Raw OCR Output: {txt}")  # Print raw OCR output
print(f"Cleaned OCR Output: {cleaned_txt}")  # Print cleaned output
print(f"OCR Time: {end_time - start_time:.2f} seconds")


Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


Raw OCR Output: Score: 11 D $ A
Cleaned OCR Output: [11, 3, 2, 1]
OCR Time: 2.52 seconds


# OCR STUFF



In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DQN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(input_dim, 16)  # First hidden layer
        self.fc2 = nn.Linear(16, 16)         # Second hidden layer
        self.fc3 = nn.Linear(16, 16)         # Second hidden layer
        self.out = nn.Linear(16, output_dim) # Output layer (Q-values)

    def forward(self, x):
        x = F.relu(self.fc1(x))   # Activation after first hidden layer
        x = F.relu(self.fc2(x))   # Activation after second hidden layer
        x = F.relu(self.fc3(x))   # Activation after second hidden layer
        return self.out(x)        # Output raw Q-values (no activation)


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import gym
import random
from collections import deque

class DQNAgent:
    def __init__(self, env, model, epsilon=0.1, gamma=0.99, lr=0.001, batch_size=64, memory_size=10000):
        self.env = env
        self.model = model
        self.target_model = DQN(env.observation_space.shape[0], env.action_space.n)
        self.target_model.load_state_dict(self.model.state_dict())  # Initialize target model with the same weights
        self.optimizer = optim.Adam(self.model.parameters(), lr=lr)
        self.criterion = nn.MSELoss()
        self.epsilon = epsilon
        self.gamma = gamma
        self.batch_size = batch_size
        self.memory = deque(maxlen=memory_size)
    
    def select_action(self, state):
        if random.random() < self.epsilon:  # Exploration: random action
            return self.env.action_space.sample()
        else:  # Exploitation: best action according to Q-network
            with torch.no_grad():
                state = torch.tensor(state, dtype=torch.float32).unsqueeze(0)  # Add batch dimension
                q_values = self.model(state)
                return torch.argmax(q_values, dim=1).item()

    def store_experience(self, experience):
        self.memory.append(experience)

    def sample_batch(self):
        batch = random.sample(self.memory, self.batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        states = torch.tensor(states, dtype=torch.float32)
        actions = torch.tensor(actions, dtype=torch.long)
        rewards = torch.tensor(rewards, dtype=torch.float32)
        next_states = torch.tensor(next_states, dtype=torch.float32)
        dones = torch.tensor(dones, dtype=torch.bool)
        return states, actions, rewards, next_states, dones

    def update_target_model(self):
        self.target_model.load_state_dict(self.model.state_dict())  # Copy weights from model to target model

    def train(self):
        if len(self.memory) < self.batch_size:
            return

        states, actions, rewards, next_states, dones = self.sample_batch()

        # Get Q-values for current states
        q_values = self.model(states)
        q_value = q_values.gather(1, actions.unsqueeze(1))

        # Get max Q-value for next states (target Q-value)
        with torch.no_grad():
            next_q_values = self.target_model(next_states)
            next_q_value = next_q_values.max(1)[0]
            target = rewards + (self.gamma * next_q_value * ~dones)

        # Compute loss and update weights
        loss = self.criterion(q_value.squeeze(1), target)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()


In [ ]:
from WASDEnv import WinFormsGameEnv
env = WinFormsGameEnv()

### states takes like 2.5 seconds to extract this way. this can be optimised to maybe 1.5s but thats still alot. this project has never been finished